In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/uci-secom.csv")
y = df["Pass/Fail"]
X = df.drop(columns=["Time", "Pass/Fail"])

print("Shape:", X.shape)
print("\nClass balance:")
print(y.value_counts())
print("\nTotal missing values:", X.isnull().sum().sum())

Shape: (1567, 590)

Class balance:
Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64

Total missing values: 41951


In [11]:
# What fraction of each column is missing?
missing_pct = X.isnull().mean().sort_values(ascending=False)

print("Top 10 most-missing columns:")
print(missing_pct.head(10))

print("\nHow many columns are >50% missing?:", (missing_pct > 0.5).sum())
print("How many columns have ZERO missing?:", (missing_pct == 0).sum())

Top 10 most-missing columns:
157    0.911934
292    0.911934
293    0.911934
158    0.911934
492    0.855775
358    0.855775
85     0.855775
220    0.855775
246    0.649649
109    0.649649
dtype: float64

How many columns are >50% missing?: 28
How many columns have ZERO missing?: 52


In [12]:
# 1. Drop columns more than 50% missing
threshold = 0.5
cols_to_drop = missing_pct[missing_pct > threshold].index
X_clean = X.drop(columns=cols_to_drop)
print("After dropping high-missing cols:", X_clean.shape)

# 2. Drop zero-variance columns (same value everywhere)
zero_var = X_clean.columns[X_clean.nunique() <= 1]
X_clean = X_clean.drop(columns=zero_var)
print("After dropping zero-variance cols:", X_clean.shape)
print("Dropped", len(zero_var), "constant columns")

# 3. Impute remaining missing values with column median
X_clean = X_clean.fillna(X_clean.median())
print("Missing values remaining:", X_clean.isnull().sum().sum())

After dropping high-missing cols: (1567, 562)
After dropping zero-variance cols: (1567, 446)
Dropped 116 constant columns
Missing values remaining: 0


In [13]:
# Reattach the label so the processed file is self-contained
processed = X_clean.copy()
processed["target"] = y
processed.to_csv("../data/processed/secom_clean.csv", index=False)
print("Saved:", processed.shape)

Saved: (1567, 447)
